In [ ]:
import subprocess
from moviepy import VideoFileClip

def robust_load_video(path, step=0.5, max_try=10.0):
    """
    Intenta cargar un video. Si falla, intenta saltar 'step' segundos 
    hasta un máximo de 'max_try'.
    """
    t_start = 0.0
    while t_start <= max_try:
        try:
            # Intentamos abrir el video. 
            # Si t_start > 0, usamos subclip para saltar el posible error inicial.
            clip = VideoFileClip(path)
            if t_start > 0:
                clip = clip.subclip(t_start)
            
            # Forzamos una lectura rápida de un frame para verificar que funciona
            clip.get_frame(0) 
            return clip, t_start
        except Exception:
            t_start += step
            print(f"  --> Fallo en t={t_start-step}s, reintentando en t={t_start}s...")
    
    return None, None

In [ ]:
# import os
# import cv2
# from moviepy import VideoFileClip
# import librosa
# import numpy as np
# from skimage.metrics import structural_similarity as ssim

# # -------------------------
# # Parámetros
# # -------------------------
# # DATA_PATH = "../data/EXIST 2026 Videos Dataset/training/videos/"
# DATA_PATH = "../data/EXIST 2026 Videos Dataset/test/videos/"
# # OUTPUT_PATH = "../data/EXIST 2026 Videos Dataset/training/important_images/"
# OUTPUT_PATH = "../data/EXIST 2026 Videos Dataset/test/important_images/"
# MIN_DISTANCE_SEC = 2
# NUM_PICOS = 3
# SSIM_THRESHOLD = 0.3
# DURATION = 2
# DARK_THRESHOLD = 30

# # -------------------------
# # Función tiempo bonito
# # -------------------------
# def format_time(seconds):
#     m = int(seconds // 60)
#     s = int(seconds % 60)
#     return f"{m:02d}:{s:02d}"

# # -------------------------
# # Procesar videos
# # -------------------------
# for path in os.listdir(DATA_PATH):
#     video_path = os.path.join(DATA_PATH, path)
#     video_name = os.path.splitext(path)[0]  # nombre del video sin extensión
#     print(f"\nProcesando video: {video_path}")

#     # Crear carpeta para guardar imágenes
#     save_dir = os.path.join(OUTPUT_PATH, video_name)
#     save_dir = os.path.join(OUTPUT_PATH, video_name)
#     if os.path.exists(save_dir):
#         if len(os.listdir(save_dir)) > 0:
#             print(f"Saltando video {video_name}, ya tiene imágenes.")
#             continue  # pasa al siguiente video
#         else:
#             print(f"Reprocesando video {video_name}, carpeta vacía.")
#     else:
#         os.makedirs(save_dir, exist_ok=True)

#     video, start_used = robust_load_video(video_path)
    
#     if video is None:
#         print(f"❌ Video {path} definitivamente ilegible tras varios intentos.")
#         continue
    
#     if start_used > 0:
#         print(f"⚠️ Video cargado saltando los primeros {start_used} segundos.")

#     # Audio
#     audio = video.audio.to_soundarray()
#     sr = video.audio.fps
#     y = audio.mean(axis=1)  # convertir a mono

#     # RMS
#     frame_length = 2048
#     hop_length = 512
#     rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]

#     # Detectar picos de audio
#     indices = np.argsort(rms)[::-1]
#     selected = []
#     min_distance = int(MIN_DISTANCE_SEC * sr / hop_length)

#     for idx in indices:
#         if all(abs(idx - s) > min_distance for s in selected):
#             selected.append(idx)
#         if len(selected) >= 10:
#             break

#     # Convertir a tiempos
#     selected_times = [(idx * hop_length / sr, rms[idx]) for idx in selected]

#     # Filtrar frames similares y descartar escenas oscuras
#     final_frames = []

#     for time_sec, rms_val in selected_times:
#         frame = video.get_frame(time_sec)
#         gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)

#         # Descartar escenas oscuras, pero permitir al menos uno
#         if gray.mean() < DARK_THRESHOLD and len(final_frames) > 0:
#             continue

#         added = True
#         for i, (existing_t, existing_frame, existing_rms) in enumerate(final_frames):
#             existing_gray = cv2.cvtColor(existing_frame, cv2.COLOR_RGB2GRAY)
#             score = ssim(gray, existing_gray)
#             if score > SSIM_THRESHOLD:
#                 if rms_val > existing_rms:
#                     final_frames[i] = (time_sec, frame, rms_val)
#                 added = False
#                 break

#         if added:
#             final_frames.append((time_sec, frame, rms_val))

#     # Si final_frames quedó vacío (todos muy oscuros), tomar el frame con mayor RMS
#     if len(final_frames) == 0 and len(selected_times) > 0:
#         time_sec, rms_val = max(selected_times, key=lambda x: x[1])
#         frame = video.get_frame(time_sec)
#         final_frames.append((time_sec, frame, rms_val))

#     # Tomar solo los primeros NUM_PICOS frames finales
#     final_frames = final_frames[:NUM_PICOS]

#     # Guardar frames como imágenes
#     for i, (time_sec, frame, _) in enumerate(final_frames):
#         save_path = os.path.join(save_dir, f"pico{i+1}.jpg")
#         frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)  # cv2 usa BGR
#         cv2.imwrite(save_path, frame_bgr)
#         print(f"Guardado: {save_path}")

In [ ]:
import os
import cv2
import librosa
import numpy as np
from moviepy import VideoFileClip
from skimage.metrics import structural_similarity as ssim
from concurrent.futures import ProcessPoolExecutor

# -------------------------
# Parámetros Globales
# -------------------------
# DATA_PATH = "../data/EXIST 2026 Videos Dataset/training/videos/"
DATA_PATH = "../data/EXIST 2026 Videos Dataset/test/videos/"
# OUTPUT_PATH = "../data/EXIST 2026 Videos Dataset/training/important_images/"
OUTPUT_PATH = "../data/EXIST 2026 Videos Dataset/test/important_images/"
MIN_DISTANCE_SEC = 2
NUM_PICOS = 3
SSIM_THRESHOLD = 0.3
DARK_THRESHOLD = 30
MAX_WORKERS = 10


def robust_load_video(path, step=0.5, max_try=2.0):
    """Intenta cargar el video saltando el inicio si está corrupto."""
    t_start = 0.0
    while t_start <= max_try:
        try:
            clip = VideoFileClip(path)
            if t_start > 0:
                clip = clip.subclip(t_start)
            clip.get_frame(0)
            return clip, t_start
        except Exception:
            t_start += step
    return None, None


def get_fallback_frame(video):
    """Extrae un único frame del centro del video como último recurso."""
    try:
        t = video.duration / 2.0
        frame = video.get_frame(t)
        return [(t, frame, 0.0)]
    except Exception:
        return []


def process_single_video(filename):
    """Función que procesa un solo video."""
    video_path = os.path.join(DATA_PATH, filename)
    video_name = os.path.splitext(filename)[0]
    save_dir = os.path.join(OUTPUT_PATH, video_name)

    # 1. Verificación de existencia
    if os.path.exists(save_dir) and len(os.listdir(save_dir)) > 0:
        return f"Saltando {video_name}: ya procesado."

    os.makedirs(save_dir, exist_ok=True)

    # 2. Carga Robusta
    video, _ = robust_load_video(video_path)
    if video is None:
        return f"Error: {video_name} es ilegible."

    try:
        final_frames = []

        # 3. Audio y RMS
        audio_clip = video.audio
        if audio_clip is None:
            # No audio — skip straight to the visual fallback below
            selected_times = []
        else:
            audio_array = audio_clip.to_soundarray()
            sr = audio_clip.fps
            y = audio_array.mean(axis=1)

            hop_length = 512
            rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop_length)[0]

            # 4. Selección de picos
            indices = np.argsort(rms)[::-1]
            selected = []
            min_distance = int(MIN_DISTANCE_SEC * sr / hop_length)

            for idx in indices:
                if all(abs(idx - s) > min_distance for s in selected):
                    selected.append(idx)
                if len(selected) >= 10:
                    break

            selected_times = [(idx * hop_length / sr, rms[idx]) for idx in selected]

        # 5. Filtrado Visual (SSIM)
        for time_sec, rms_val in selected_times:
            if time_sec >= video.duration:
                continue

            frame = video.get_frame(time_sec)
            gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)

            # Skip dark frames only when we already have at least one saved
            if gray.mean() < DARK_THRESHOLD and len(final_frames) > 0:
                continue

            added = True
            for i, (existing_t, existing_frame, existing_rms) in enumerate(final_frames):
                existing_gray = cv2.cvtColor(existing_frame, cv2.COLOR_RGB2GRAY)
                if gray.shape != existing_gray.shape:
                    existing_gray = cv2.resize(existing_gray, (gray.shape[1], gray.shape[0]))

                score = ssim(gray, existing_gray)
                if score > SSIM_THRESHOLD:
                    if rms_val > existing_rms:
                        final_frames[i] = (time_sec, frame, rms_val)
                    added = False
                    break

            if added:
                final_frames.append((time_sec, frame, rms_val))

        # 6. Fallback: guarantee at least one frame is always present
        if len(final_frames) == 0:
            final_frames = get_fallback_frame(video)
            if not final_frames:
                return f"Error: {video_name} — no se pudo extraer ningún frame."

        # 7. Guardado
        final_frames = final_frames[:NUM_PICOS]
        for i, (time_sec, frame, _) in enumerate(final_frames):
            save_path = os.path.join(save_dir, f"pico{i+1}.jpg")
            cv2.imwrite(save_path, cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

        # Limpieza CRÍTICA para evitar fugas de memoria y procesos zombie
        if video.audio:
            video.audio.close()
        video.close()

        return f"Éxito: {video_name} procesado."

    except Exception as e:
        return f"Error procesando {video_name}: {str(e)}"


# -------------------------
# Ejecución Principal
# -------------------------
if __name__ == "__main__":
    if not os.path.exists(OUTPUT_PATH):
        os.makedirs(OUTPUT_PATH)

    all_videos = [f for f in os.listdir(DATA_PATH) if f.endswith(('.mp4', '.avi', '.mov'))]

    print(f"Iniciando procesamiento en paralelo para {len(all_videos)} videos...")
    print(f"Usando {MAX_WORKERS} procesos simultáneos.")

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(executor.map(process_single_video, all_videos))

    for res in results:
        print(res)